In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

# To keep this script runnable without the file, 
# I am generating a local miniature version of the dataset.
raw_data = {
    'PassengerId': [1, 2, 3, 4, 5, 6, 7],
    'Survived': [0, 1, 1, 1, 0, 0, 0],
    'Pclass': [3, 1, 3, 1, 3, 3, 1],
    'Sex': ['male', 'female', 'female', 'female', 'male', 'male', 'male'],
    'Age': [22.0, 38.0, 26.0, 35.0, 35.0, np.nan, 54.0],
    'SibSp': [1, 1, 0, 1, 0, 0, 0],
    'Parch': [0, 0, 0, 0, 0, 0, 0],
    'Fare': [7.25, 71.28, 7.92, 53.1, 8.05, 8.45, 51.86]
}
df = pd.DataFrame(raw_data)


# PRE-PROCESSING
df['Age'] = df['Age'].fillna(df['Age'].median())

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1


# CORRELATION HEATMAP
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.show()


sns.boxplot(x=df['Fare'])
plt.show()

start_lambda = time.time()

def process_age(x):
    if x < 18:
        return 'Child'
    elif x < 35:
        return 'Young Adult'
    elif x < 60:
        return 'Adult'
    else:
        return 'Senior'

df['age_bin_lambda'] = df['Age'].apply(process_age)
end_lambda = time.time()
lambda_time = end_lambda - start_lambda

start_cut = time.time()
df['age_bin'] = pd.cut(df['Age'], bins=[0, 18, 35, 60, 100], labels=['Child', 'Young Adult', 'Adult', 'Senior'])
end_cut = time.time()
cut_time = end_cut - start_cut

print("Lambda technique time:", lambda_time)
print("Pandas cut technique time:", cut_time)

age_groups = ['Child', 'Young Adult', 'Adult', 'Senior']
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for i in range(len(age_groups)):
    group = age_groups[i]
    subset = df[df['age_bin'] == group]
    axes[i].hist(subset['Fare'], bins=10, color='skyblue')
    axes[i].set_title(group)
    
plt.tight_layout()
plt.show()
